# EXP-AKAMAI-CONCURRENCY-001 · Agentic × antigentic pilot

**One-click Akamai experiment** — benchmark four **proxy orchestration policies** against the *same* synthetic deterministic ECA tasks with **per-arm concurrency 1 and 2**, plus read-only vLLM metric snapshots. The GPU is not reconfigured. Run all cells after uploading this notebook and `CELL_AKAMAI_CONCURRENCY_KIT.zip` to the same JupyterLab folder.

**Evidence ceiling:** nonconfirmatory ECA workload only. B1/B2 are Genesis/Vita DAISY *proxy strategies*, **not** verified original trained systems. All MMR, signature, Anticube, ΔG* statuses remain uncommitted/uncomputed. Real output tokens are taken from response `usage.completion_tokens` — **never counted from stream chunks**. Run states and failures are retained in a downloadable ZIP.

Default mode = `AKAMAI_LIVE`. For an explicitly labeled local synthetic fixture run only, set `CELL_AKAMAI_BACKEND=MOCK_FIXTURE` before starting the kernel.


## 1. Extract the upload and bind the fixed taskset


In [ ]:
import os, sys, zipfile, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import FileLink, display
ROOT=Path.cwd()/"cell_akamai_concurrency_kit"
ZIP=Path.cwd()/"CELL_AKAMAI_CONCURRENCY_KIT.zip"
if not (ROOT/"scripts/experiment.py").exists():
    assert ZIP.is_file(), f"Upload {ZIP.name} beside this notebook, or extract the kit first. CWD={Path.cwd()}"
    with zipfile.ZipFile(ZIP) as z:
        bad=z.testzip()
        assert bad is None, f"ZIP CRC failure {bad}"
        for name in z.namelist():
            target=(Path.cwd()/name).resolve()
            assert target.is_relative_to(Path.cwd().resolve()), f"Unsafe ZIP member {name}"
        z.extractall(Path.cwd())
assert (ROOT/"data/pilot_items.jsonl").exists()
sys.path.insert(0,str(ROOT))
from scripts.experiment import run_sweep, ARMS
from scripts.score import score_run
from scripts.checkpoint import produce
from scripts.export import export_run
records=[json.loads(x) for x in (ROOT/"data/pilot_items.jsonl").read_text().splitlines()]
print("Taskset:",len(records),"items; rules:",sorted({x['rule'] for x in records}))
print("Taskset source:", ROOT/"data/DATASET_MANIFEST.json")


## 2. Choose bounded real GPU or explicit mock fixture execution


In [ ]:
from urllib.request import urlopen, Request
BACKEND=os.environ.get("CELL_AKAMAI_BACKEND","AKAMAI_LIVE")
assert BACKEND in ("AKAMAI_LIVE","MOCK_FIXTURE")
CONCURRENCIES=(1,2)  # bounded initial pilot; broaden only after inspection
MAX_ITEMS=8         # 2 per ECA rule, not 8 from one rule
MAX_OUTPUT_TOKENS=96
PILOT_REPEAT=1
SELECTED=[]
for rule in sorted({r['rule'] for r in records}):
    SELECTED.extend([r for r in records if r['rule']==rule][:MAX_ITEMS//4])
assert len(SELECTED)==MAX_ITEMS and len({x['item_id'] for x in SELECTED})==MAX_ITEMS
if BACKEND=="MOCK_FIXTURE":
    from scripts.mock_endpoint import start
    _mock_server, ENDPOINT=start()
    MODEL="MOCK_ECA_FIXTURE"; SOURCE_CLASS="SYNTHETIC_FIXTURE"
    print("WARNING: MOCK_FIXTURE — no Akamai inference is being measured")
else:
    ENDPOINT=os.getenv("VLLM_HOST","http://vllm:8000/v1").rstrip("/")
    if not ENDPOINT.endswith('/v1'):ENDPOINT+='/v1'
    MODEL_DATA=json.load(urlopen(Request(ENDPOINT+"/models"),timeout=20))
    models=[x['id'] for x in MODEL_DATA['data']]
    assert models, "No served models available"
    MODEL=models[0]
    SOURCE_CLASS="OBSERVED"
    print("Served live models:",models)
    print("Configured model env:",os.getenv('MODEL_NAME','<missing>'))
print("Backend:",BACKEND,"model:",MODEL,"endpoint:",ENDPOINT)
print("Experimental workload:",len(SELECTED),"items × 4 arms × 1 or 3 calls × 2 concurrency levels")


## 3. Execute independent arm blocks with item-level parallelism

Each arm runs in a separate window. Increasing item concurrency does **not** mix competing arms on the GPU; this reduces inter-arm interference. Order is randomized deterministically per level. Raw Prometheus probes are read-only.


In [ ]:
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir=Path.cwd()/"cell_akamai_results"/f"EXP-AKAMAI-CONCURRENCY-001_{BACKEND}_{stamp}"
try:
    index=run_sweep(SELECTED,run_dir,endpoint=ENDPOINT,model=MODEL,
        concurrencies=CONCURRENCIES,max_items=MAX_ITEMS,max_tokens=MAX_OUTPUT_TOKENS,
        repeat=PILOT_REPEAT,source_class=SOURCE_CLASS,backend=BACKEND)
finally:
    if BACKEND=="MOCK_FIXTURE":_mock_server.shutdown();_mock_server.server_close()
print("Results:",run_dir)
print("Blocks:",len(index['results']))
print("Statuses:",{r['status'] for r in index['results']})
assert any(r['status']=='OBSERVED' for r in index['results']), "No successful block; inspect failures before promoting any claim"


## 4. Independently re-derive ECA outcomes and preserve evidence


In [ ]:
score_rows=score_run(run_dir/'RUN_INDEX.json', ROOT/'data/pilot_items.jsonl', ROOT/'data/pilot_oracle.jsonl')
from collections import Counter
print('Scored blocks:',len(score_rows))
for row in sorted(score_rows,key=lambda x:(x['concurrency'],x['arm'])):
    print(f"c={row['concurrency']} {row['arm']:<19} accuracy={row['exact_match_accuracy']:.3f} "
          f"tok/s={str(round(row['tokens_per_s'],2)) if row['tokens_per_s'] is not None else 'UNKNOWN':>8} "
          f"items/s={row['items_per_s']:.3f} failures={row['failures']}")
local_checkpoint=produce(run_dir)
assert local_checkpoint['verification']=='LOCAL_RECOMPUTE_PASS'
print('Local Merkle root (not project MMR):',local_checkpoint['local_root_sha256'])
print('Project MMR:',local_checkpoint['project_mmr_state'])


## 5. Visual comparison (same benchmark taskset; nonconfirmatory)


In [ ]:
import matplotlib.pyplot as plt
series=sorted(set(x['concurrency'] for x in score_rows))
fig,ax=plt.subplots(figsize=(10,4))
for arm in ARMS:
    rr=sorted((x for x in score_rows if x['arm']==arm),key=lambda x:x['concurrency'])
    ax.plot([x['concurrency'] for x in rr],[x['correct_items_per_s'] for x in rr],marker='o',label=arm)
ax.set(xlabel='Concurrent task instances per arm',ylabel='Correct items / second',title='Agentic vs antigentic policy proxies — ECA pilot')
ax.grid(alpha=.3);ax.legend(fontsize=8,ncol=2);fig.tight_layout();plt.show()
fig,ax=plt.subplots(figsize=(10,4))
for arm in ARMS:
    rr=sorted((x for x in score_rows if x['arm']==arm),key=lambda x:x['concurrency'])
    ax.plot([x['concurrency'] for x in rr],[x['tokens_per_s'] for x in rr],marker='o',label=arm)
ax.set(xlabel='Concurrent task instances per arm',ylabel='Server-reported completion tokens / s',title='Throughput: do not compare alone without task correctness')
ax.grid(alpha=.3);ax.legend(fontsize=8,ncol=2);fig.tight_layout();plt.show()


## 6. Export evidence for ChatGPT and Claude

The verified ZIP link below is authenticated to your JupyterLab session. Review any response contents before sharing. SHA-256 and local Merkle integrity do not imply scientific validity or trusted external attestation.


In [ ]:
RESULT_ZIP=Path.cwd()/"CELL_AKAMAI_CONCURRENCY_RESULTS.zip"
export_receipt=export_run(run_dir,RESULT_ZIP)
print(json.dumps(export_receipt,indent=2))
print("Akamai source class:",SOURCE_CLASS)
print("Anticube: NOT_COMPUTED; ΔG*: NOT_COMPUTED; MMR: NOT_COMMITTED; signature: NOT_SIGNED")
display(FileLink(str(RESULT_ZIP)))
